# FCF — Обучение на Wikipedia (Google Colab)

**Единая Вычислительная Архитектура**  
GPU-обучение языковой модели с нуля на русской Википедии

### Результат за 2–4 часа (T4, бесплатно):
- Токенизатор: 50K слов на Wikipedia
- Языковая модель: 50K шагов Causal LM (~150 млн токенов)
- Осмысленная генерация русского текста

In [ ]:
# @title 1. Установка зависимостей
!pip install -q torch numpy faiss-cpu tokenizers datasets scipy loguru
print('Зависимости установлены')

In [ ]:
# @title 2. Клонирование репозитория
import os
if not os.path.exists('FCF'):
    !git clone https://github.com/BlackCatSpb/FCF.git
    print('Репозиторий склонирован')
else:
    %cd FCF
    !git pull
    %cd ..
    print('Репозиторий обновлён')
%cd FCF

In [ ]:
# @title 3. Проверка GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA доступен: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_vram:.1f} GB VRAM)')
    can_fit = gpu_vram >= 1.5
    print(f'Модель помещается: {"ДА" if can_fit else "НЕТ (нужно >1.5 GB)"}')

In [ ]:
# @title 4. Токенизатор на Wikipedia (50K слов, ~20 мин)
import sys, os
if not os.path.exists('fcf'): %cd FCF
sys.path.insert(0, '.')
from fcf.tokenizer_utils import train_tokenizer_on_wikipedia

print('Обучение BPE-токенизатора на русской Википедии...')
print('100 000 статей, vocab_size=50257')

tokenizer = train_tokenizer_on_wikipedia(
    output_path='tokenizer_wiki.json',
    vocab_size=50257,
    num_texts=100000,
    min_text_length=100,
)

if tokenizer:
    print(f'Токенизатор: {tokenizer.get_vocab_size()} токенов')
    test = tokenizer.encode('История это наука о прошлом')
    print(f'Тест: {len(test.ids)} токенов')
else:
    print('ОШИБКА')

In [ ]:
# @title 5. PrimordialLayer (233M параметров)
import sys, os
if not os.path.exists('fcf'): %cd FCF
sys.path.insert(0, '.')
from fcf.config import load_config
from fcf.primordial_layer import PrimordialLayer

config = load_config()
layer = PrimordialLayer(config)

params = sum(p.numel() for p in layer.parameters())
print(f'Создан: {params:,} параметров ({params*4/1e9:.2f} GB FP32)')

test_input = torch.randint(0, 1000, (1, 16))
x = layer.embed(test_input)
h = layer.forward_transformer(x)
l = layer.forward_logits(h)
print(f'Прямой проход: embed={x.shape} -> hidden={h.shape} -> logits={l.shape}')

In [ ]:
# @title 6. Пункт 2 — Causal LM на Wikipedia
# @markdown ~3000 tok/s на T4, 50K шагов = 2.5 часа

import sys, os
if not os.path.exists('fcf'): %cd FCF
sys.path.insert(0, '.')
from fcf.language_trainer import LanguageTrainer
from fcf.tokenizer_utils import load_tokenizer
import torch

tokenizer = load_tokenizer('tokenizer_wiki.json')

MAX_STEPS = 50000
DEVICE = 'cuda'

trainer = LanguageTrainer(
    layer=layer,
    tokenizer=tokenizer,
    checkpoint_dir='checkpoints/language_wiki',
)

print(f'Обучение: {MAX_STEPS} шагов, Wikipedia streaming, GPU')
stats = trainer.train(
    max_steps=MAX_STEPS,
    device=DEVICE,
    use_wikipedia=True,
)
print(f'Результат: avg_conf={stats.get("average_confidence", 0):.3f}')

In [ ]:
# @title 7. Сохранить чекпоинт
import sys, os
if not os.path.exists('fcf'): %cd FCF
sys.path.insert(0, '.')
from fcf.utils import save_primordial_layer
import os, json

path = 'checkpoints/language_wiki/final'
save_primordial_layer(layer, path)

files = os.listdir(path)
print(f'Сохранено: {path}')
print(f'Файлы: {files}')

stats = {
    'params': sum(p.numel() for p in layer.parameters()),
    'confidence': layer.meta.average_confidence(),
    'snapshots': len(layer.state_storage),
}
with open(f'{path}/stats.json', 'w') as f:
    json.dump(stats, f)
print(f'Статистика: {stats}')

In [ ]:
# @title 8. Скачать чекпоинт
!zip -r fcf_wiki_checkpoint.zip checkpoints/language_wiki/final tokenizer_wiki.json tokenizer_wiki_vocab.json
from google.colab import files
files.download('fcf_wiki_checkpoint.zip')
print('Чекпоинт скачан. Распакуйте в папку FCF на своём ПК.')

In [ ]:
# @title 9. Тест генерации (бонус)
layer.eval()
device = next(layer.parameters()).device

prompts = [
    'История — это наука',
    'Математика изучает',
    'Человек отличается от животных',
]

for prompt in prompts:
    ids = tokenizer.encode(prompt).ids
    inp = torch.tensor([ids], dtype=torch.long).to(device)
    out = layer.generate(inp, max_new_tokens=30, temperature=0.8)
    resp = tokenizer.decode(out[0].tolist())
    print(f'Q: {prompt}')
    print(f'A: {resp}')
    print()